In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import os
from dataset import get_val_tiles_auto, get_llto_splits

In [ ]:
PROCESSED_DIR = "/scratch/sloeblein/postprocessed"
CSV_PATH = "../final_processing_pipeline/data/train_test_split.csv"
K_FOLDS = 4

In [ ]:
all_folds = get_llto_splits(PROCESSED_DIR, CSV_PATH, k=K_FOLDS)

In [ ]:
val_files = all_folds[0][1]

In [ ]:
def verify_tiling(cube_paths, patch_size=256, dim_max=1000):
    """
    Visualisiert die Patch-Strategie für den ersten Cube in der Liste.
    """
    # 1. Generiere die komplette Liste über alle Cubes
    tiled_list = get_val_tiles_auto(cube_paths, patch_size, dim_max)

    # 2. Filtere nur die Tiles für den ERSTEN Cube heraus
    first_cube_path = cube_paths[0]
    first_cube_tiles = [tile for tile in tiled_list if tile["path"] == first_cube_path]

    # 3. Plotting Vorbereitung
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.set_xlim(0, dim_max)
    ax.set_ylim(dim_max, 0) # 0,0 ist oben links (Standard für Bilder)
    ax.set_aspect('equal')
    
    num_tiles_per_side = int(np.sqrt(len(first_cube_tiles)))
    ax.set_title(f"Tiling Check: {num_tiles_per_side}x{num_tiles_per_side} Patches\n"
                 f"Cube: {os.path.basename(first_cube_path)} | Patch Size: {patch_size}")

    # 4. Zeichne die Patches ein
    # Wir nutzen einen Farbübergang, um die Reihenfolge der Patches zu sehen
    colors = plt.cm.plasma(np.linspace(0, 1, len(first_cube_tiles)))

    for idx, tile in enumerate(first_cube_tiles):
        top = tile["top"]
        left = tile["left"]
        
        # Rechteck (x, y, width, height)
        # In Matplotlib ist x=left, y=top
        rect = patches.Rectangle(
            (left, top), patch_size, patch_size, 
            linewidth=1.5, edgecolor=colors[idx], 
            facecolor=colors[idx], alpha=0.25, label='_nolegend_'
        )
        ax.add_patch(rect)
        
        # Nummerierung zur Kontrolle der Reihenfolge (Wichtig für Rekonstruktion!)
        ax.text(left + patch_size/2, top + patch_size/2, str(idx), 
                fontsize=9, color='black', ha='center', va='center', fontweight='bold')

    # Hilfslinien zur Orientierung
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.xlabel("Width (X pixels)")
    plt.ylabel("Height (Y pixels)")
    
    # Kleiner Hinweis zur Überlappung
    print(f"Visualisiere {len(first_cube_tiles)} Tiles für den ersten Cube.")
    plt.show()

# Beispielaufruf (angenommen val_files ist deine Liste mit Pfaden)
verify_tiling(val_files, patch_size=128, dim_max=1000)

In [ ]:
files = os.listdir(PROCESSED_DIR)
path = PROCESSED_DIR + "/" + files[2]

In [ ]:
import xarray as xr#
import pandas as pd
ds = xr.open_zarr(path, consolidated=True)
ds.dims
context_len = 20
target_len = 5

In [ ]:
cutoff_date = pd.to_datetime(ds.attrs["precip_end_date"])
print(cutoff_date)
# ds_ctx: (time_context, y, x)
ds_ctx = ds.sel(time_sentinel_2_l2a=slice(None, cutoff_date)).tail(
    time_sentinel_2_l2a=context_len
)
# ds_target: (time_target, y, x)
ds_target = ds.sel(
    time_sentinel_2_l2a=slice(cutoff_date + pd.Timedelta(days=1), None)
).head(time_sentinel_2_l2a=target_len)

# Asserts for Time Length
assert (
    len(ds_ctx.time_sentinel_2_l2a) == context_len
), f"Context time mismatch: expected {context_len}, got {len(ds_ctx.time_sentinel_2_l2a)}"
assert (
    len(ds_target.time_sentinel_2_l2a) == target_len
), f"Target time mismatch: expected {target_len}, got {len(ds_target.time_sentinel_2_l2a)}"
assert (
    pd.to_datetime(ds_ctx.time_sentinel_2_l2a[-1].values) == cutoff_date)
assert (
    pd.to_datetime(ds_target.time_sentinel_2_l2a[0].values) == cutoff_date + pd.Timedelta(days=5))

In [ ]:
cutoff_date = pd.to_datetime(ds.attrs["precip_end_date"])

# cutoff_date must be an available timestep in the series
full_time = pd.to_datetime(ds.time_sentinel_2_l2a.values)
if cutoff_date not in full_time:
    raise ValueError(
        f"cutoff_date={cutoff_date} is not an available timestep. Path: {path}"
    )

# ds_ctx: (time_context, y, x)
# Includes cutoff_date
ds_ctx = ds.sel(time_sentinel_2_l2a=slice(None, cutoff_date)).tail(
    time_sentinel_2_l2a=context_len
)

# ds_target: (time_target, y, x)
ds_target = ds.where(ds.time_sentinel_2_l2a > cutoff_date, drop=True).head(
    time_sentinel_2_l2a=target_len
        )


In [ ]:
print(cutoff_date)
print(ds_ctx.time_sentinel_2_l2a[-1].values)
print(ds_target.time_sentinel_2_l2a[0].values)

In [ ]:
era5 = ["t2m_mean", "t2mmax_max", "t2mmin_min", # Temperature
    "tp_dailymean_mean", "tp_dailymax_max",        # Precipitation
    "pei_30_mean", "pei_90_mean"]
len(era5)

In [ ]:
## test llto, broadcast and onehot

In [ ]:
import torch
import torch.nn.functional as F
def encode_landcover(lc_tensor):
    """
    Transforms ESA Landcover into One-Hot Encoding.
    Input: (T, H, W) or (H, W)
    Output: (T, 12, H, W)
    """
    labels = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 100]
    mapping = torch.zeros(101, dtype=torch.long)
    for i, val in enumerate(labels):
        mapping[val] = i

    lc_mapped = mapping[lc_tensor]
    # One-Hot and permute to (T, C, H, W)
    lc_onehot = F.one_hot(lc_mapped, num_classes=len(labels))

    if lc_onehot.ndim == 4:  # (T, H, W, C)
        return lc_onehot.permute(0, 3, 1, 2).float()
    else:  # (H, W, C) -> (C, H, W)
        return lc_onehot.permute(2, 0, 1).float()


In [ ]:
patch_size = 256
x_fut_era5_1d = torch.from_numpy(ds_target[era5].to_array().values).float()
from dataset import broadcast_era5
x_fut_era5 = broadcast_era5(x_fut_era5_1d, patch_size, patch_size)
print(x_fut_era5_1d.shape, x_fut_era5.shape)

In [ ]:
context_kndvi = ds_ctx.kNDVI.values
context_kndvi.shape

In [ ]:
x_fut_era5[1][0]

In [ ]:
x_fut_era5[1][0] == x_fut_era5[1][4]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold
from matplotlib.patches import Patch

def get_llto_splits(root_dir, csv_path="train_test_split.csv", k=3, show=False):
    """
    Implements Leave-Location-and-Time-Out (LLTO) Cross-Validation according to
    https://doi.org/10.1016/j.envsoft.2017.12.001.
    
    This strategy prevents spatio-temporal over-fitting by ensuring that 
    locations and time units present in the validation set are strictly 
    excluded from the training set within each fold.

    Groups data by Koppen-Geiger region (Location) and Phenological Season (Time)
    and ensures that regions and seasons in the validation set are not present in the training set.
    """
    # 1. Load metadata and filter training split
    df = pd.read_csv(csv_path)
    df = df[df["split"] == "train"].copy()

    # 2. Map file IDs to full disk paths
    processed_files = {
        f.replace("_postprocessed.zarr", ""): os.path.join(root_dir, f)
        for f in os.listdir(root_dir) if f.endswith(".zarr")
    }
    df["full_path"] = df["DisNo."].map(processed_files)
    df = df.dropna(subset=["full_path"])

    # Identify unique locations and time units (LLTO) 
    locations = df["climate_class"].unique()
    seasons = df["pheno_season_name"].unique()

    # Build LLTO groups (combination of location and season)
    df["llto_group"] = df["climate_class"].astype(str) + "_" + df["pheno_season_name"].astype(str)
    unique_groups = df["llto_group"].unique()


    # Vorbereitung für den Plot
    group_map = {grp: i for i, grp in enumerate(unique_groups)}
    
    from sklearn.model_selection import KFold
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    cv_splits = []
    
    # Visualisierungs-Matrix: (Folds x Gruppen)
    # 0 = Train, 1 = Validation
    vis_matrix = np.zeros((k, len(unique_groups)))

    for fold_idx, (train_grp_idx, val_grp_idx) in enumerate(kf.split(unique_groups)):
        # 1. Bestimme, welche GRUPPEN in die Validierung gehen
        val_groups_names = unique_groups[val_grp_idx]
        
        # 2. Identifiziere die betroffenen Locations und Zeiten der Validierung
        val_df = df[df["llto_group"].isin(val_groups_names)]
        val_locations = val_df["koppen_geiger"].unique()
        val_seasons = val_df["pheno_season_name"].unique()

        # 3. Training-Set erstellen:
        # Schließe alle aus, die die gleiche Location ODER die gleiche Zeit wie die Validierung haben
        train_df = df[
            (~df["koppen_geiger"].isin(val_locations)) & 
            (~df["pheno_season_name"].isin(val_seasons))
        ]

        train_paths = train_df["full_path"].tolist()
        val_paths = val_df["full_path"].tolist()

        cv_splits.append((train_paths, val_paths))
        
        print(f"Fold {fold_idx}: Val-Groups={len(val_groups_names)}, "
              f"Train-Cubes={len(train_paths)}, Val-Cubes={len(val_paths)}")

    # --- Visualisierung ---
    if show:
        plt.figure(figsize=(12, 6))
        # Wir plotten die Gruppen-Zugehörigkeit
        sns.heatmap(vis_matrix, annot=False, cbar=False, cmap=["#f0f0f0", "#e74c3c"], 
                    linewidths=1, linecolor='white')
        
        plt.title(f"LLTO-CV Splitting Strategy ({k} Folds)")
        plt.xlabel("Unique Groups (Location-Time Units)")
        plt.ylabel("Fold Index")
        
        # Legende händisch hinzufügen
        from matplotlib.patches import Patch
        legend_elements = [Patch(facecolor='#f0f0f0', edgecolor='gray', label='Training'),
                           Patch(facecolor='#e74c3c', edgecolor='gray', label='Validation')]
        plt.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(1.2, 1))
        
        # Gruppen-Labels auf X-Achse (optional, kann bei vielen Gruppen unübersichtlich sein)
        if len(unique_groups) < 20:
            plt.xticks(np.arange(len(unique_groups))+0.5, unique_groups, rotation=90)
        
        plt.tight_layout()
        plt.show()

    return cv_splits

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold
from matplotlib.patches import Patch

def get_llto_splits(root_dir, csv_path="train_test_split.csv", k=3, show=False):
    """
    Implements Leave-Location-and-Time-Out (LLTO) Cross-Validation.
    
    This strategy prevents spatio-temporal over-fitting by ensuring that 
    locations and time units present in the validation set are strictly 
    excluded from the training set within each fold.

    Args:
        root_dir (str): Path to the directory containing .zarr cubes.
        csv_path (str): Path to the metadata CSV file.
        k (int): Number of folds for cross-validation.
        show (bool): If True, displays a visualization of the splitting strategy.

    Returns:
        list: A list of tuples [(train_paths, val_paths), ...] for k folds.
    """
    # 1. Load metadata and filter training split
    df = pd.read_csv(csv_path)
    df = df[df["split"] == "train"].copy()

    # 2. Map file IDs to full disk paths
    processed_files = {
        f.replace("_postprocessed.zarr", ""): os.path.join(root_dir, f)
        for f in os.listdir(root_dir) if f.endswith(".zarr")
    }
    df["full_path"] = df["DisNo."].map(processed_files)
    df = df.dropna(subset=["full_path"])

    # 3. Define LLTO groups (Location-Time combinations)
    # Using koppen_geiger for Location and pheno_season_name for Time
    df["llto_group"] = (
        df["koppen_geiger"].astype(str) + "_" + 
        df["pheno_season_name"].astype(str)
    )
    unique_groups = df["llto_group"].unique()

    # 4. Initialize K-Fold on the unique Group IDs
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    cv_splits = []
    
    # visualization matrix: 0=Excluded/Blocked, 1=Training, 2=Validation
    vis_matrix = np.zeros((k, len(unique_groups)))
    group_map = {grp: i for i, grp in enumerate(unique_groups)}

    for fold_idx, (_, val_grp_idx) in enumerate(kf.split(unique_groups)):
        # Identify specific Groups, Locations, and Seasons for Validation
        val_group_names = unique_groups[val_grp_idx]
        val_df = df[df["llto_group"].isin(val_group_names)]
        
        val_locations = val_df["koppen_geiger"].unique()
        val_seasons = val_df["pheno_season_name"].unique()

        # LLTO Logic: Exclude any data sharing the same Location OR same Season
        # This prevents the model from 'seeing' the spatial or temporal 
        # characteristics of the validation targets during training.
        train_df = df[
            (~df["koppen_geiger"].isin(val_locations)) & 
            (~df["pheno_season_name"].isin(val_seasons))
        ]

        train_paths = train_df["full_path"].tolist()
        val_paths = val_df["full_path"].tolist()
        cv_splits.append((train_paths, val_paths))

        # Update visualization matrix
        # First, mark everything that could be training
        for grp in unique_groups:
            vis_matrix[fold_idx, group_map[grp]] = 1
            
        # Second, mark excluded groups (same Loc or same Time as Val)
        excluded_mask = (
            df["koppen_geiger"].isin(val_locations) | 
            df["pheno_season_name"].isin(val_seasons)
        )
        excluded_groups = df[excluded_mask]["llto_group"].unique()
        for grp in excluded_groups:
            vis_matrix[fold_idx, group_map[grp]] = 0
            
        # Third, mark validation groups
        for grp in val_group_names:
            vis_matrix[fold_idx, group_map[grp]] = 2

        print(f"Fold {fold_idx}: Val-Groups={len(val_group_names)}, "
              f"Train-Cubes={len(train_paths)}, Val-Cubes={len(val_paths)}")

    # 5. Optional Visualization
    if show:
        _plot_llto_strategy(vis_matrix, unique_groups, k)

    return cv_splits

def _plot_llto_strategy(vis_matrix, unique_groups, k):
    """Internal helper to plot the LLTO CV strategy matrix."""
    plt.figure(figsize=(14, 6))
    
    # Map: 0: Grey (Excluded), 1: LightBlue (Train), 2: Red (Val)
    colors = ["#bdc3c7", "#3498db", "#e74c3c"]
    sns.heatmap(vis_matrix, annot=False, cbar=False, cmap=colors, 
                linewidths=1, linecolor='white')
    
    plt.title(f"Target-Oriented LLTO-CV Strategy ({k} Folds)")
    plt.xlabel("Spatio-Temporal Groups (Location_Season)")
    plt.ylabel("Fold Index")
    
    legend_elements = [
        Patch(facecolor="#3498db", label='Training'),
        Patch(facecolor="#e74c3c", label='Validation'),
        Patch(facecolor="#bdc3c7", label='Excluded (Same Loc/Time as Val)')
    ]
    plt.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(1.25, 1))
    
    if len(unique_groups) < 30:
        plt.xticks(np.arange(len(unique_groups)) + 0.5, unique_groups, rotation=90)
    
    plt.tight_layout()
    plt.show()

In [ ]:
cv_splits = get_llto_splits(PROCESSED_DIR, csv_path=CSV_PATH, k=3, show=True)